# Analyse des Similarity-Threshold-Experiments (Translation: SFT mBART-50 LoRA)

Dieses Notebook analysiert und vergleicht systematisch den Einfluss des **minimalen semantischen Ähnlichkeits-Schwellenwerts** ($s_{\min} \in \{0.60, 0.70, 0.80\}$ bei $s_{\max} = 0.98$) auf das **Supervised Fine-Tuning (SFT)** von `facebook/mbart-large-50`:

1. **Trainingsdynamik & Konvergenz:** Lernkurven (Train vs. Val Loss), Early-Stopping-Verhalten und Trainingsdauer.
2. **Out-of-Domain Generierungsqualität (Lebenshilfe-Benchmark):**
   * Stilistische Einfachheit ($R_{\text{style}}$ gemessen via BiLSTM MixUp Regressor)
   * Semantischer Erhalt zur Ausgangssprache ($R_{\text{sem, AS}}$ via Jina SBERT)
   * Ähnlichkeit zur menschlichen Referenzübersetzung ($Sim_{\text{ref}}$)
   * Verbund-Belohnung ($\text{Composite Reward} = 0{,}5 \cdot R_{\text{style}} + 0{,}5 \cdot R_{\text{sem}}$)
3. **Lexikalische & Syntaktische Indikatoren:** BLEU, ROUGE-L, Satzabbruch-Quote (*Truncation Rate* in %) und Tokenlängen.
4. **Qualitative Textanalyse:** Direkte Gegenüberstellung generierter Übersetzungen für konkrete Testdokumente.
5. **LaTeX-Tabellen-Export:** Formatierte Ergebnistabellen für Kapitel 5 (*Experiments & Results*) der Masterarbeit.

---

In [ ]:
import os
import sys
import json
import glob
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown, HTML

def find_repo_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, 'data')) and os.path.exists(os.path.join(p, 'results')):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.expanduser('~/Documents/Master Thesis'))

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print('Arbeitsverzeichnis:', os.getcwd())

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'figure.dpi': 150
})

RESULTS_DIR = os.path.join(REPO_ROOT, 'results/experiments/similarity_threshold')
SUMMARY_CSV = os.path.join(RESULTS_DIR, 'similarity_threshold_summary.csv')
DETAILS_CSV = os.path.join(RESULTS_DIR, 'similarity_threshold_details.csv')


## 1. SFT Evaluationsdaten laden

Einlesen aller aggregierten Metriken der trainierten SFT-Modelle (0.60, 0.70, 0.80).

In [ ]:
sft_metrics_files = sorted(glob.glob(os.path.join(RESULTS_DIR, 'sft_sim_*_metrics.json')))
sft_records = []

for f in sft_metrics_files:
    with open(f, 'r', encoding='utf-8') as jf:
        sft_records.append(json.load(jf))

if sft_records:
    df_sft = pd.DataFrame(sft_records).sort_values(by='min_sim')
    display(Markdown('### SFT Translationsmodelle Metriken-Übersicht:'))
    cols = ['min_sim', 'num_train_pairs', 'best_val_loss', 'r_style_mean', 'r_sem_as_mean', 'sim_ref_mean', 'composite_reward_mean', 'bleu_mean', 'rouge_l_mean', 'truncation_rate_pct', 'avg_gen_tokens']
    available_cols = [c for c in cols if c in df_sft.columns]
    display(df_sft[available_cols])
else:
    print('Keine SFT Metriken gefunden. Bitte Trainingsskripte auf dem Cluster ausführen.')
    df_sft = pd.DataFrame()


## 2. Quantitative Evaluation: Reward-Metriken & Pareto-Frontier

Untersuchung des Trade-offs zwischen stilistischer Einfachheit ($R_{\text{style}}$) und inhaltlicher Semantik ($R_{\text{sem}}$).

In [ ]:
if not df_sft.empty and len(df_sft) >= 2:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Subplot 1: Reward Komponenten über Schwellenwerte
    ax1.plot(df_sft['min_sim'], df_sft['r_style_mean'], marker='o', color='#e7298a', linewidth=2.5, label='Einfachheit ($R_{style}$)')
    ax1.plot(df_sft['min_sim'], df_sft['r_sem_as_mean'], marker='s', color='#1f78b4', linewidth=2.5, label='Semantik zu AS ($R_{sem}$)')
    ax1.plot(df_sft['min_sim'], df_sft['composite_reward_mean'], marker='D', color='#33a02c', linewidth=2.5, linestyle='--', label='Composite Reward (0.5/0.5)')
    ax1.set_xlabel('Minimaler Schwellenwert $s_{min}$', fontweight='bold')
    ax1.set_ylabel('Reward / Score [0.0 - 1.0]', fontweight='bold')
    ax1.set_title('SFT Reward-Metriken auf Lebenshilfe', fontweight='bold')
    ax1.legend(loc='best')
    
    # Subplot 2: Lexikalische Metriken vs. Truncation
    ax2.plot(df_sft['min_sim'], df_sft['bleu_mean'], marker='o', color='#ff7f00', linewidth=2, label='BLEU')
    ax2.plot(df_sft['min_sim'], df_sft['rouge_l_mean'], marker='s', color='#6a3d9a', linewidth=2, label='ROUGE-L')
    ax2.set_xlabel('Minimaler Schwellenwert $s_{min}$', fontweight='bold')
    ax2.set_ylabel('Score', fontweight='bold')
    ax2.set_title('Lexikalische Überlappung (BLEU & ROUGE-L)', fontweight='bold')
    ax2.legend(loc='best')
    
    plt.tight_layout()
    plt.show()


## 3. Detailanalysen & Satzabbruchquoten (*Truncation Rate*)

Analyse der generierten Ausgabelängen und Satzabbrüche in Abhängigkeit des Filterbereichs.

In [ ]:
if not df_sft.empty and 'truncation_rate_pct' in df_sft.columns:
    fig, ax1 = plt.subplots(figsize=(8, 4.5))
    
    c1 = '#d62728'
    ax1.set_xlabel('Minimaler Schwellenwert $s_{min}$', fontweight='bold')
    ax1.set_ylabel('Satzabbruchquote Truncation (%)', color=c1, fontweight='bold')
    ax1.plot(df_sft['min_sim'], df_sft['truncation_rate_pct'], marker='o', color=c1, linewidth=2.5, label='Truncation Rate')
    ax1.tick_params(axis='y', labelcolor=c1)
    
    ax2 = ax1.twinx()
    c2 = '#2b5c8f'
    ax2.set_ylabel('Ø Generierte Tokens', color=c2, fontweight='bold')
    ax2.plot(df_sft['min_sim'], df_sft['avg_gen_tokens'], marker='s', color=c2, linestyle='--', linewidth=2, label='Ø Tokens')
    ax2.tick_params(axis='y', labelcolor=c2)
    
    plt.title('Satzabbruch-Quote und Textlänge über Schwellenwerte', fontweight='bold', pad=15)
    fig.tight_layout()
    plt.show()


## 4. Qualitativer Inferenz-Vergleich auf Lebenshilfe-Artikeln

Side-by-Side Gegenüberstellung konkreter Modellübersetzungen.

In [ ]:
sft_detail_files = sorted(glob.glob(os.path.join(RESULTS_DIR, 'sft_sim_*_details.csv')))
if sft_detail_files:
    dfs = [pd.read_csv(f) for f in sft_detail_files]
    df_comp = pd.concat(dfs, ignore_index=True)
    
    display(Markdown(f'### Geladene Detail-Übersetzungen: {len(df_comp)} Zeilen'))
    
    # Zeige Beispiel 0
    sample_idx = 0
    if len(dfs[0]) > sample_idx:
        as_ex = dfs[0].iloc[sample_idx]['as_text']
        ref_ex = dfs[0].iloc[sample_idx]['ls_ref_text']
        
        display(Markdown(f'**Original Alltagssprache (AS):**\n> {as_ex[:400]}...'))
        display(Markdown(f'**Menschliche Referenz (LS):**\n> {ref_ex[:400]}...'))
        
        for df_d in dfs:
            exp_n = df_d.iloc[sample_idx]['experiment_name']
            gen_t = df_d.iloc[sample_idx]['generated_text']
            r_st = df_d.iloc[sample_idx].get('r_style', 0)
            r_se = df_d.iloc[sample_idx].get('r_sem_as', 0)
            display(Markdown(f'**Modell {exp_n}** ($R_{{style}}={r_st:.3f}, R_{{sem}}={r_se:.3f}$):\n> {gen_t[:400]}...'))
else:
    print('Keine Detail-CSVs gefunden.')


## 5. LaTeX Tabellen Generator (Kapitel 5: Experiments & Results)

Formatierter LaTeX-Code zur direkten Einbindung in `thesis/5.experimentsAndResults/results.tex`.

In [ ]:
print('% ==============================================================================')
print('% LaTeX Tabelle: SFT mBART-50 nach Similarity-Schwellenwerten')
print('% ==============================================================================')
print('\\begin{table}[htbp]')
print('\\centering\\small')
print('\\caption{Vergleich der SFT-Modellgüte auf dem Lebenshilfe-Benchmark ($N=37$).}')
print('\\label{tab:sft_similarity_ablation}')
print('\\begin{tabular}{@{}lrrrrrrr@{}}')
print('\\toprule')
print('\\textbf{Filter} & \\textbf{$R_{style}$} & \\textbf{$R_{sem, AS}$} & \\textbf{$Sim_{ref}$} & \\textbf{Composite} & \\textbf{BLEU} & \\textbf{ROUGE-L} & \\textbf{Trunc.} \\\\')
print('\\midrule')

if not df_sft.empty:
    for _, r in df_sft.iterrows():
        filt = f"${r['min_sim']:.2f} \\le s \\le 0{,}98$"
        r_st = f"{r.get('r_style_mean', 0):.4f}"
        r_se = f"{r.get('r_sem_as_mean', 0):.4f}"
        sim_r = f"{r.get('sim_ref_mean', 0):.4f}"
        comp = f"{r.get('composite_reward_mean', 0):.4f}"
        bleu = f"{r.get('bleu_mean', 0):.4f}"
        rouge = f"{r.get('rouge_l_mean', 0):.4f}"
        trunc = f"{r.get('truncation_rate_pct', 0):.1f}\\,\\%"
        print(f"{filt} & {r_st} & {r_se} & {sim_r} & {comp} & {bleu} & {rouge} & {trunc} \\\\")

print('\\bottomrule')
print('\\end{tabular}')
print('\\end{table}')
